# prime-rl S2: reverse-text SFT smoke (RTX 6000 Pro offline)

Goal: confirm `prime-rl sft` runs end-to-end on the offline RTX 6000 Pro using
the wheels + Qwen3-0.6B model + reverse-text dataset bundled by
`prime-rl-offline-prep`.

Tiny config (`max_steps=5 batch=1 seq=512`) so we just verify the pipeline,
not training quality.

Success: SFT loop completes 5 steps without error and produces a checkpoint.

## 1. Install prime-rl (same recipe as S1)

In [ ]:
import os, subprocess, glob, re

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
WHEELS = f"{BASE}/wheels"
MODEL_DIR = f"{BASE}/models/Qwen3-0.6B"
DATASET_DIR = f"{BASE}/datasets/reverse-text"

# Skip the GPU stack -- system already has matching torch / NCCL
SKIP_PREFIXES = ("torch-", "torchvision-", "torchaudio-", "nvidia_")

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m:
        return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]

keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w)
        continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)

filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")

subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)

## 2. Sanity check

In [ ]:
import subprocess, os

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/Qwen3-0.6B"
DATASET_DIR = f"{BASE}/datasets/reverse-text"

subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv", shell=True)
print()
subprocess.run(f"ls {MODEL_DIR}", shell=True)
print()
subprocess.run(f"ls {DATASET_DIR}", shell=True)
print()
import torch, prime_rl, transformers
print(f"torch={torch.__version__} cuda_ok={torch.cuda.is_available()}")
print(f"transformers={transformers.__version__}")
print(f"prime_rl={getattr(prime_rl, '__version__', '?')}")

## 3. Write minimal SFT TOML

In [ ]:
import os

PREP = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
MODEL_DIR = f"{BASE}/models/Qwen3-0.6B"
DATASET_DIR = f"{BASE}/datasets/reverse-text"

# prime-rl SFTConfig.ckpt does not accept a `path` field -- the example
# reverse_text/sft.toml ships an empty `[ckpt]` block. Trust the default
# checkpoint location and just enable checkpointing by declaring the section.
TOML = f'''max_steps = 5

[model]
name = "{MODEL_DIR}"

[data]
name = "{DATASET_DIR}"
seq_len = 512
batch_size = 1

[optim]
lr = 2e-5

[ckpt]
'''
os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/sft.toml", "w") as f:
    f.write(TOML)
import subprocess
subprocess.run("cat /kaggle/working/sft.toml", shell=True)

## 4. Run SFT (5 steps)

Force every loader to operate offline so a stray HF / W&B network call cannot
hang the run.

In [ ]:
import os, subprocess, time

env = os.environ.copy()
env["HF_HUB_OFFLINE"] = "1"
env["TRANSFORMERS_OFFLINE"] = "1"
env["HF_DATASETS_OFFLINE"] = "1"
env["WANDB_MODE"] = "disabled"

t0 = time.time()
result = subprocess.run(
    "python3.12 -m prime_rl.entrypoints.sft @ /kaggle/working/sft.toml",
    shell=True, env=env,
)
elapsed = time.time() - t0
print(f"\nsft exit={result.returncode}  elapsed={elapsed:.1f}s")
if result.returncode != 0:
    raise RuntimeError(f"SFT failed with exit code {result.returncode}")

# Show whatever the trainer wrote
subprocess.run("ls -la /kaggle/working/sft_ckpt 2>&1 || true", shell=True)
print("\nS2 PASS")